# 💰 options_premia — are some options systematically mispriced?

**Research only.** The first study (`options_hedge.ipynb`) found that options on "outlier" days are
priced about right. This notebook tests three other ways to earn from trading options themselves,
pre-registered in `README.md` ("Second study: option premia"):

- **A · Cheap vs dear options**: buy straddles whose implied volatility is low against the stock's
  own history, sell those where it is high (Goyal & Saretto 2009).
- **B · Earnings options**: sell the straddle the evening before a release and buy it back after it
  (`crush`); buy it a week before and sell it the evening before (`runup`).
- **C · Plain premium selling**: write 5%-out-of-the-money puts, or sell at-the-money straddles,
  every day, as a base rate.

Every trade pays the real spread: bought at the ask, sold at the bid, $0.65 per contract.
Straddles use the liquid strike (`rs.LIQUID`). Holdout locked behind `READ_HOLDOUT`.

In [ ]:
# If this import fails: select the "trading-simple" Jupyter kernel (README "Quick start")
import importlib.util
import json

import numpy as np
import pandas as pd
import plotly.graph_objects as go

from trading_simple import plots, strategies


def _load(folder, name):
    spec = importlib.util.spec_from_file_location(name, strategies.STRATEGIES_DIR / folder
                                                  / "research.py")
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod


rs = _load("options_hedge", "options_hedge_research")
os_rs = _load("option_signals", "option_signals_research")

# --- the pre-registered windows (README) ---------------------------------------------------
DISC = dict(months=("2025-10", "2026-04"), first_entry="2025-10-15", last_close="2026-04-30")
HOLD = dict(months=("2026-05", "2026-08"), first_entry="2026-05-01", last_close="2026-08-31")
READ_HOLDOUT = False           # True reads the locked window ONCE, for discovery-passing cells only
N_DRAWS = 20_000
FEE = rs.FEE / 100             # $ per share of one contract
pd.set_option("display.width", 170)

## 1 · Data

One function builds everything a window needs, so the holdout runs the same code:
- the first study's chain summaries (liquid straddles, splits, SEC earnings dates);
- `os_rs` implied volatilities (ATM IV, the 0.95 put) and a year of daily bars before them;
- the window's eligible straddle rows with both signals of test A read on the screen day t.

In [ ]:
def build(win, extra_months=None):
    first_m, last_m = win["months"]
    m0 = extra_months or first_m
    summ = rs.chain_summaries(m0, last_m)
    symbols = sorted(p.name for p in rs.CHAINS.iterdir() if p.is_dir())
    daily = rs.load_daily(sorted(summ["symbol"].unique()), "2025-08-01", win["last_close"])
    sessions = rs.calendar(summ, daily)
    earnings = rs.fetch_earnings(sorted(summ["symbol"].unique()))
    D = rs.assemble(summ, daily, pd.DataFrame(index=sessions), earnings, sessions)
    odaily = os_rs.load_daily(symbols, "2024-09-01", win["last_close"])
    oidx = pd.DatetimeIndex(sorted(odaily["date"].unique()))
    ocols = sorted(odaily["symbol"].unique())
    px = {c: os_rs.wide(odaily, c, oidx, ocols) for c in ("close", "adj_close", "volume")}
    events = os_rs.dividend_events(px["close"], px["adj_close"])
    surf = os_rs.surfaces(m0, last_m, events, symbols)
    iv = (os_rs.wide(surf, "iv_call", oidx, ocols) + os_rs.wide(surf, "iv_put", oidx, ocols)) / 2
    hv = np.log(px["adj_close"]).diff().rolling(252, min_periods=252).std() * np.sqrt(252)
    sig = {"hv_iv": np.log(hv / iv),
           "iv_own": np.log(iv / iv.shift(1).rolling(20, min_periods=20).median())}
    w = rs.window_rows(D["rows"]["30d_liq"], D["react"], sessions, win["first_entry"],
                       win["last_close"])
    for k, f in sig.items():
        w[k] = rs._at(f.reindex(index=sessions, columns=sorted(summ["symbol"].unique())),
                      w["t_pos"].to_numpy(), w["symbol"].to_numpy())
    bid = w["cb"] + w["pb"]
    w["r_sell"] = (100 * bid - 2 * rs.FEE - 100 * w["payoff"] - rs.FEE * (w["payoff"] > 0)) \
        / (100 * bid)
    w["tercile"] = 1                       # test A matches on the date only
    return dict(summ=summ, daily=daily, sessions=sessions, D=D, surf=surf, px=px, iv=iv, w=w,
                symbols=symbols, oidx=oidx, ocols=ocols)


B = build(DISC)
W = B["w"]
print(f"30-day liquid straddle rows in the window: {len(W):,}; eligible (no earnings): "
      f"{int(W['elig'].sum()):,} on {W.loc[W['elig'], 'symbol'].nunique()} names")
print("signals present on eligible rows:",
      {k: round(float(W.loc[W["elig"], k].notna().mean()), 3) for k in ("hv_iv", "iv_own")})

## 2 · A · Cheap vs dear options

Each entry day, among eligible straddles with the signal: the cheapest tenth (high `hv_iv` / low
`iv_own`) is **bought at the ask**; the dearest tenth is **sold at the bid**. Each is compared with
20,000 random sets of eligible straddles with the same number per entry date (`ctrl`), bought or
sold the same way. `p` = share of random sets that did at least as well.
**Pass**: mean > 0, p ≤ 0.006, and > 0 without the best week and without the top-5 names.

In [ ]:
CELLS_A = {("hv_iv", "buy cheap"): ("hv_iv", True, "r_ask"),
           ("hv_iv", "sell dear"): ("hv_iv", False, "r_sell"),
           ("iv_own", "buy cheap"): ("iv_own", False, "r_ask"),
           ("iv_own", "sell dear"): ("iv_own", True, "r_sell")}
ALPHA_A = 0.006


def test_a(w, cells=CELLS_A):
    out = {}
    for key, (col, top, value) in cells.items():
        pool = w[w["elig"] & w[col].notna()].reset_index(drop=True)
        pct = pool.groupby("entry")[col].rank(pct=True)
        ev = pool[pct > 0.9] if top else pool[pct <= 0.1]
        r = rs.score_cell(pool, ev, n_draws=N_DRAWS, seed=len(out), value=value)
        r["passed"] = bool(r["r_ask"] > 0 and r["p"] <= ALPHA_A and r["robust"])
        r["implied"], r["ctrl_implied"] = ev["implied"].mean(), pool["implied"].mean()
        out[key] = r
    return out


def table_a(res):
    cols = ["n", "names", "weeks", "r_ask", "ctrl_r_ask", "excess", "excess_lo", "excess_hi", "p",
            "p_names", "no_best_week", "no_top5", "implied", "ctrl_implied", "passed"]
    t = pd.DataFrame({k: {c: r[c] for c in cols} for k, r in res.items()}).T
    return t.rename(columns={"r_ask": "mean", "ctrl_r_ask": "ctrl"})


RA = test_a(W)
TA = table_a(RA)
print(TA.to_string(float_format=lambda x: f"{x:.4f}"))

## 3 · B · Earnings options

Every SEC Item 2.02 release whose reaction session r is inside the window. The straddle is the liquid
ATM one of the first expiry on or after r (strike within 5%, spread ≤ 10% at entry); the exit marks
the same two contracts.
- `crush`: sold at the bid at r − 1's close, bought back at the ask at r's close;
- `runup`: bought at the ask at r − 6's close, sold at the bid at r − 1's close.

Return per $ of premium, $0.65 per contract per trade. `p_names` / `p_weeks`: one-sided p of a mean
≤ 0 when whole names / whole entry weeks are resampled. **Pass**: mean > 0 with both p ≤ 0.025, and
> 0 without the top-5 names. Reading ~1,500 names' chains takes several minutes the first time
(cached in `var/options_hedge/`).

In [ ]:
def earnings_trades(b, win, tag):
    path = rs.CACHE / f"earn_trades_{tag}.pkl"
    if path.exists():
        return pd.read_pickle(path)
    s = b["sessions"]
    react = b["D"]["react"].drop_duplicates()
    lo = s.searchsorted(pd.Timestamp(win["first_entry"]))
    hi = s.searchsorted(pd.Timestamp(win["last_close"]), side="right")
    plans = []
    for trade, (a, z) in {"crush": (-1, 0), "runup": (-6, -1)}.items():
        ok = react[(react["pos"] + a >= lo) & (react["pos"] + z < hi) & (react["pos"] < len(s))]
        plans.append(pd.DataFrame({"symbol": ok["symbol"], "trade": trade,
                                   "entry": s[ok["pos"] + a], "exit": s[ok["pos"] + z],
                                   "min_exp": s[ok["pos"]]}))
    plans = pd.concat(plans, ignore_index=True)
    out = []
    for k, (sym, g) in enumerate(plans.groupby("symbol"), 1):
        out.append(rs.event_straddles(sym, g.drop(columns="symbol")))
        if k % 200 == 0:
            print(f"\r{k}/{plans['symbol'].nunique()} names", end="")
    out = pd.concat([o for o in out if len(o)], ignore_index=True)
    out.to_pickle(path)
    return out


def test_b(trades):
    t = trades.dropna(subset=["cb1", "ca1", "pb1", "pa1"]).copy()
    sold, bought = t["cb0"] + t["pb0"], t["ca0"] + t["pa0"]
    crush = t["trade"] == "crush"
    t["ret"] = np.where(crush, (sold - t["ca1"] - t["pa1"] - 4 * FEE) / sold,
                        (t["cb1"] + t["pb1"] - bought - 4 * FEE) / bought)
    t["move"] = (t["S1"] / t["S0"] - 1).abs()
    t["priced"] = ((t["cb0"] + t["ca0"] + t["pb0"] + t["pa0"]) / 2) / t["S0"]
    res = {}
    for trade, g in t.groupby("trade"):
        week = g["entry"].dt.to_period("W-FRI").astype(str)
        top = g.groupby("symbol")["ret"].sum().nlargest(5).index
        r = dict(n=len(g), names=g["symbol"].nunique(), mean=g["ret"].mean(),
                 median=g["ret"].median(), win=(g["ret"] > 0).mean(),
                 worst_1pct=g["ret"].quantile(0.01), p_names=rs.bootstrap_p(g["ret"], g["symbol"]),
                 p_weeks=rs.bootstrap_p(g["ret"], week),
                 no_top5=g.loc[~g["symbol"].isin(top), "ret"].mean(),
                 move=g["move"].mean(), priced=g["priced"].mean())
        r["passed"] = bool(r["mean"] > 0 and r["p_names"] <= 0.025 and r["p_weeks"] <= 0.025
                           and r["no_top5"] > 0)
        res[trade] = r
    return pd.DataFrame(res).T, t


TR = earnings_trades(B, DISC, "disc")
TB, TRB = test_b(TR)
print(f"\nreleases with a liquid straddle at entry: {len(TR):,}; "
      f"both legs quoted at exit: {len(TRB):,}")
print(TB.to_string(float_format=lambda x: f"{x:.4f}"))

## 4 · C · Plain premium selling (base rate)

- `put_write`: every session, sell the expiry-A put nearest 95% of the price at the bid (names in the
  `os_rs` universe, put spread ≤ 20% of mid), hold to expiry, cash-secured. Earnings included.
- `short_straddle`: every eligible 30-day liquid straddle (no earnings in the hold) sold at the bid.

Return per $ of premium. **Pass**: mean > 0 with p ≤ 0.025 resampling whole entry weeks, and > 0
without the top-5 names. `worst_1pct` is the price of the premium.

In [ ]:
def put_writes(b, win):
    surf = b["surf"]
    o = b["px"]
    dvol = o["close"] * o["volume"]
    S = os_rs.wide(surf, "S", b["oidx"], b["ocols"])
    uni = os_rs.universe(b["iv"], S, dvol).stack()
    uni = uni[uni].index.to_frame(index=False)
    uni.columns = ["date", "symbol"]
    x = surf.merge(uni, on=["date", "symbol"]).dropna(subset=["K_otm_put", "otm_put_bid"])
    mid = (x["otm_put_bid"] + x["otm_put_ask"]) / 2
    x = x[(x["otm_put_bid"] > 0) & ((x["otm_put_ask"] - x["otm_put_bid"]) / mid <= 0.20)]
    x = x[(x["date"] >= pd.Timestamp(win["first_entry"]))
          & (x["exp"] <= pd.Timestamp(win["last_close"]))]
    st = surf.set_index(["symbol", "date"])["S"]
    x["ST"] = st.reindex(pd.MultiIndex.from_arrays([x["symbol"], x["exp"]])).to_numpy()
    x = x.dropna(subset=["ST"])
    pay = (x["K_otm_put"] - x["ST"]).clip(lower=0)
    x["ret"] = (x["otm_put_bid"] - FEE - pay) / x["otm_put_bid"]
    x["entry"] = x["date"]
    return x


def base_rate(x, value="ret"):
    week = x["entry"].dt.to_period("W-FRI").astype(str)
    top = x.groupby("symbol")[value].sum().nlargest(5).index
    r = dict(n=len(x), names=x["symbol"].nunique(), mean=x[value].mean(),
             median=x[value].median(), win=(x[value] > 0).mean(),
             worst_1pct=x[value].quantile(0.01), p_weeks=rs.bootstrap_p(x[value], week),
             no_top5=x.loc[~x["symbol"].isin(top), value].mean())
    r["passed"] = bool(r["mean"] > 0 and r["p_weeks"] <= 0.025 and r["no_top5"] > 0)
    return r


PW = put_writes(B, DISC)
SS = W[W["elig"]]
TC = pd.DataFrame({"put_write": base_rate(PW), "short_straddle": base_rate(SS, "r_sell")}).T
print(f"put positions: {len(PW):,} on {PW['symbol'].nunique()} names; "
      f"median premium {(PW['otm_put_bid'] / PW['K_otm_put']).median():.2%} of the strike, "
      f"assigned (finished in the money) {(PW['ST'] < PW['K_otm_put']).mean():.1%}")
print(TC.to_string(float_format=lambda x: f"{x:.4f}"))

## 4b · Exploratory: before costs, and A's two legs together (NOT part of the verdict)

- **At the mid price**: the same positions with no spread paid. A mispricing that exists at the mid
  but not after the spread is real but not tradable by a retail account.
- **A, both legs at once** (Goyal & Saretto's long-short): buy the cheap tenth and sell the dear tenth
  on the same days; excess over random = the two cells' excesses added, its z from their random-set
  spreads (independent draws).
- **B, near expiries only**: the liquid-strike rule let half the earnings straddles drift to expiries
  weeks or years out (median 23 days; the event is then a small part of the price); here only
  expiries within 14 days of the release.

In [ ]:
mid = {}
for key, (col, top, value) in CELLS_A.items():
    pool = W[W["elig"] & W[col].notna()]
    pct = pool.groupby("entry")[col].rank(pct=True)
    ev = pool[pct > 0.9] if top else pool[pct <= 0.1]
    sgn = 1 if value == "r_ask" else -1          # a seller earns minus the buyer's mid return
    mid[key] = {"at mid": sgn * ev["r_mid"].mean(), "random at mid": sgn * pool["r_mid"].mean(),
                "after costs": RA[key]["r_ask"], "random after costs": RA[key]["ctrl_r_ask"]}
print("A at the mid price vs after costs (per $ of premium):")
print(pd.DataFrame(mid).T.round(4).to_string())
for sig in ("hv_iv", "iv_own"):
    a, b = RA[(sig, "buy cheap")], RA[(sig, "sell dear")]
    sd = np.hypot((a["excess_hi"] - a["excess"]) / 1.96, (b["excess_hi"] - b["excess"]) / 1.96)
    print(f"A {sig}: buy cheap + sell dear, excess over random {a['excess'] + b['excess']:+.4f} "
          f"(z {(a['excess'] + b['excess']) / sd:.2f})")

e = TRB.copy()
mid0 = (e["cb0"] + e["ca0"] + e["pb0"] + e["pa0"]) / 2
mid1 = (e["cb1"] + e["ca1"] + e["pb1"] + e["pa1"]) / 2
e["ret_mid"] = np.where(e["trade"] == "crush", (mid0 - mid1) / mid0, (mid1 - mid0) / mid0)
e["near"] = (e["exp"] - e["min_exp"]).dt.days <= 14
e["spread_in"] = (e["ca0"] + e["pa0"] - e["cb0"] - e["pb0"]) / mid0
e["spread_out"] = (e["ca1"] + e["pa1"] - e["cb1"] - e["pb1"]) / mid1
print("\nB at the mid price, all expiries and near ones:")
print(e.groupby(["trade", "near"]).agg(n=("ret", "size"), after_costs=("ret", "mean"),
                                       at_mid=("ret_mid", "mean"),
                                       spread_in=("spread_in", "median"),
                                       spread_out=("spread_out", "median")).round(4).to_string())

## 5 · Picture

Mean return per $ of premium for every cell (A against its random-days control).

In [ ]:
rows = [(f"A {k[0]} {k[1]}", r["r_ask"], r["ctrl_r_ask"]) for k, r in RA.items()]
rows += [(f"B {k}", v["mean"], 0.0) for k, v in TB.iterrows()]
rows += [(f"C {k}", v["mean"], 0.0) for k, v in TC.iterrows()]
fig = go.Figure()
fig.add_trace(go.Bar(x=[r[0] for r in rows], y=[r[1] for r in rows], name="trade",
                     marker={"color": plots.SERIES[0]},
                     hovertemplate="%{x}<br>%{y:+.1%}<extra>trade</extra>"))
fig.add_trace(go.Scatter(x=[r[0] for r in rows], y=[r[2] for r in rows], mode="markers",
                         name="random days (A) / zero", marker={"color": plots._MUTED, "size": 10,
                                                                 "symbol": "line-ew-open",
                                                                 "line": {"color": plots._MUTED,
                                                                          "width": 2}},
                         hovertemplate="%{x}<br>control %{y:+.1%}<extra></extra>"))
fig.add_hline(y=0, line={"color": plots._INK, "width": 1})
fig.update_yaxes(tickformat="+.0%", title_text="mean return per $ of premium")
plots._base(fig, title="Option premia, discovery window", height=440)
fig.update_layout(legend={"orientation": "h", "yanchor": "top", "y": -0.3, "x": 0,
                          "xanchor": "left"}, margin={"b": 130})
fig.show()

## 6 · Verdict

In [ ]:
CHECKS = pd.DataFrame([
    ("A: both signals on ≥ 80% of eligible rows",
     str({k: round(float(W.loc[W['elig'], k].notna().mean()), 2) for k in ("hv_iv", "iv_own")}),
     all(W.loc[W["elig"], k].notna().mean() >= 0.8 for k in ("hv_iv", "iv_own"))),
    ("B: ≥ 80% of earnings trades quoted at exit", f"{len(TRB) / max(len(TR), 1):.1%}",
     len(TRB) / max(len(TR), 1) >= 0.8),
    ("B: the straddle spans the release (expiry ≥ reaction session)",
     str(bool((TR["exp"] >= TR["min_exp"]).all())), bool((TR["exp"] >= TR["min_exp"]).all())),
    ("C: put outcomes found", f"{len(PW):,}", len(PW) > 1000),
], columns=["check", "value", "ok"])
print(CHECKS.to_string(index=False))
passing = ([("A",) + k for k, r in RA.items() if r["passed"]]
           + [("B", k) for k, v in TB.iterrows() if v["passed"]]
           + [("C", k) for k, v in TC.iterrows() if v["passed"]])
print(f"\n{len(passing)} of {len(RA) + len(TB) + len(TC)} cells pass discovery: {passing or 'none'}"
      + ("" if passing else " — the holdout stays unread."))
SCORECARD = {"notebook": "options_premia", "window": "discovery",
             "checks_ok": bool(CHECKS["ok"].all()),
             "A": {f"{k[0]}/{k[1]}": {"n": int(r["n"]), "mean": round(r["r_ask"], 4),
                                      "ctrl": round(r["ctrl_r_ask"], 4), "p": round(r["p"], 4),
                                      "passed": r["passed"]} for k, r in RA.items()},
             "B": {k: {"n": int(v["n"]), "mean": round(v["mean"], 4),
                       "p_names": round(v["p_names"], 4), "p_weeks": round(v["p_weeks"], 4),
                       "passed": bool(v["passed"])} for k, v in TB.iterrows()},
             "C": {k: {"n": int(v["n"]), "mean": round(v["mean"], 4),
                       "p_weeks": round(v["p_weeks"], 4), "passed": bool(v["passed"])}
                   for k, v in TC.iterrows()},
             "passing": ["/".join(p) for p in passing]}
print("SCORECARD " + json.dumps(SCORECARD))

## 7 · Holdout (locked)

Positions opened 2026-05-01 on and closed by 2026-08-31, scored ONCE, only for cells that passed
discovery, with the same bar except α divided by the number of such cells. The holdout chains are
read only here (April kept for the signals' trailing windows).

In [ ]:
if not passing:
    print("Nothing passed discovery: the holdout stays unread.")
elif not READ_HOLDOUT:
    print(f"Holdout locked (READ_HOLDOUT = False); it would score {passing}.")
else:
    m = len(passing)
    BH = build(HOLD, extra_months="2026-04")
    out = {}
    for cell in passing:
        if cell[0] == "A":
            ALPHA_A = 0.006 / m
            r = test_a(BH["w"], {cell[1:]: CELLS_A[cell[1:]]})[cell[1:]]
            out["/".join(cell)] = {"n": int(r["n"]), "mean": r["r_ask"], "ctrl": r["ctrl_r_ask"],
                                   "p": r["p"], "pass": r["passed"]}
        elif cell[0] == "B":
            tb, _ = test_b(earnings_trades(BH, HOLD, "hold"))
            v = tb.loc[cell[1]]
            out["/".join(cell)] = {"n": int(v["n"]), "mean": v["mean"], "p_names": v["p_names"],
                                   "p_weeks": v["p_weeks"],
                                   "pass": bool(v["mean"] > 0 and max(v["p_names"], v["p_weeks"])
                                                <= 0.025 / m)}
        else:
            x = put_writes(BH, HOLD) if cell[1] == "put_write" else BH["w"][BH["w"]["elig"]]
            v = base_rate(x, "ret" if cell[1] == "put_write" else "r_sell")
            out["/".join(cell)] = {"n": v["n"], "mean": v["mean"], "p_weeks": v["p_weeks"],
                                   "pass": bool(v["mean"] > 0 and v["p_weeks"] <= 0.025 / m)}
    print(pd.DataFrame(out).T.to_string())
    print("SCORECARD " + json.dumps({"notebook": "options_premia", "window": "holdout",
                                     "cells": {k: {kk: (round(float(vv), 4)
                                                        if isinstance(vv, float) else vv)
                                                   for kk, vv in v.items()}
                                               for k, v in out.items()}}, default=str))

## 7b · Holdout confirmation of the one exploratory lead

Pre-registered in `README.md` after discovery and before the holdout was read: test A's `hv_iv`
with both legs at once (buy the cheapest tenth at the ask, sell the dearest tenth at the bid).
**Confirmed** if the combined excess over random is > 0 with z ≥ 1.65 and each leg's excess > 0.
The holdout was read for this test once, on 2026-10-19; it is now spent for test A, which is why the
switch stays on (it reproduces that one read).

In [ ]:
READ_HOLDOUT_LEAD = True       # spent 2026-10-19 for this one hypothesis (README)
LEAD = [("hv_iv", "buy cheap"), ("hv_iv", "sell dear")]
if READ_HOLDOUT_LEAD:
    BH = build(HOLD, extra_months="2026-04")
    WH = BH["w"]
    print(f"holdout 30-day liquid straddles: {len(WH):,}; eligible {int(WH['elig'].sum()):,} "
          f"on {WH.loc[WH['elig'], 'symbol'].nunique()} names, entries "
          f"{WH['entry'].min():%Y-%m-%d} → {WH['entry'].max():%Y-%m-%d}")
    RH = test_a(WH, {k: CELLS_A[k] for k in LEAD})
    a, b = RH[LEAD[0]], RH[LEAD[1]]
    sd = np.hypot((a["excess_hi"] - a["excess"]) / 1.96, (b["excess_hi"] - b["excess"]) / 1.96)
    z = (a["excess"] + b["excess"]) / sd
    confirmed = bool(z >= 1.65 and a["excess"] > 0 and b["excess"] > 0)
    print(table_a(RH).drop(columns="passed").to_string(float_format=lambda x: f"{x:.4f}"))
    print(f"\ncombined excess {a['excess'] + b['excess']:+.4f}, z {z:.2f} → "
          f"{'CONFIRMED' if confirmed else 'NOT confirmed'}")
    print("SCORECARD " + json.dumps({"notebook": "options_premia", "window": "holdout",
                                     "lead": "A hv_iv long-short",
                                     "excess_buy": round(a["excess"], 4),
                                     "excess_sell": round(b["excess"], 4),
                                     "z": round(float(z), 2), "confirmed": confirmed}))